# Phase 4 — Pac-Man A* Pathfinding

This notebook implements an **automated Pac-Man game using A\***.

### Assignment objectives covered

- A* pathfinding with `f(n) = g(n) + h(n)`.
- `Direction` enum with four cardinal movements.
- Wall detection.
- Ghost avoidance.
- Repeated food collection until all current food is eaten.
- Automated play — **no keyboard input**.
- A small 3×3 A* test before the full game.
- A randomized animated game on every demo run.

## A* idea

For each candidate cell,

\[
f(n) = g(n) + h(n)
\]

- `g(n)` = exact path cost from Pac-Man to the cell.
- `h(n)` = Manhattan distance from the cell to the target food.
- `f(n)` = estimated total path cost.

The open set is a min-heap. A `best_costs` map stores the best known `g` value,
and a `parents` map reconstructs the final route.

Because ghosts move, Pac-Man **replans every game step** instead of following
one stale path for the whole game.

## Architecture

```mermaid
flowchart TD
    A[Random game state] --> B[Ghost safety zone]
    B --> C[Choose reachable food]
    C --> D[A* search]
    D --> E[Take first path step]
    E --> F[Collect food]
    F --> G[Move ghosts]
    G --> H{All food collected?}
    H -- No --> B
    H -- Yes --> I[Win]
```

The search algorithm does not know anything about Matplotlib. Rendering,
search, safety, game state, and control are separate modules.

### Constants

In [1]:
MAZE_LAYOUT: tuple[str, ...] = (
    "#############",
    "#.....#.....#",
    "#.###.#.###.#",
    "#.#.......#.#",
    "#.#.#####.#.#",
    "#...........#",
    "###.#.#.#.###",
    "#...........#",
    "#.#.#####.#.#",
    "#.#.......#.#",
    "#.###.#.###.#",
    "#.....#.....#",
    "#############",
)

WALL_CELL = "#"
OPEN_CELL = "."
INFINITE_COST = 10**9

DEFAULT_FOOD_COUNT = 12
DEFAULT_GHOST_COUNT = 2
DEFAULT_MAX_STEPS = 320
MIN_SPAWN_DISTANCE = 3
EVALUATION_GAMES = 20

FOOD_SCORE = 10
STEP_SCORE = -1
DEATH_SCORE = -100

ANIMATION_INTERVAL_MS = 180
FIGURE_SIZE = (7, 7)
PACMAN_SIZE = 520
GHOST_SIZE = 420
FOOD_SIZE = 35
WALL_SIZE = 900

COLOR_WALL = "#172554"
COLOR_FOOD = "#facc15"
COLOR_PACMAN = "#facc15"
COLOR_GHOST = "#ef4444"
COLOR_TEXT = "#111827"


### Enums

In [2]:
from enum import Enum


class Direction(Enum):
    NORTH = (-1, 0)
    SOUTH = (1, 0)
    WEST = (0, -1)
    EAST = (0, 1)


class GameStatus(Enum):
    RUNNING = "running"
    WON = "won"
    LOST = "lost"


### Data Models

In [3]:
from __future__ import annotations
from dataclasses import dataclass

Cell = tuple[int, int]


@dataclass(frozen=True)
class SearchResult:
    goal: Cell
    path: tuple[Cell, ...]
    explored: frozenset[Cell]
    cost: int


@dataclass(frozen=True)
class SearchTask:
    maze: Maze
    goal: Cell
    blocked: frozenset[Cell]


@dataclass(frozen=True)
class Edge:
    parent: Cell
    neighbor: Cell


@dataclass(frozen=True)
class GameConfig:
    food_count: int
    ghost_count: int
    max_steps: int


@dataclass(frozen=True)
class GameState:
    pacman: Cell
    ghosts: tuple[Cell, ...]
    foods: frozenset[Cell]
    score: int
    steps: int
    status: GameStatus


@dataclass(frozen=True)
class EvaluationSummary:
    games: int
    wins: int
    losses: int
    success_rate: float
    mean_steps: float
    max_steps: int


### Maze Representation

In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Maze:
    layout: tuple[str, ...]

    def is_inside(self, cell: Cell) -> bool:
        row, column = cell
        row_ok = 0 <= row < len(self.layout)
        column_ok = 0 <= column < len(self.layout[0])
        return all((row_ok, column_ok))

    def is_open(self, cell: Cell) -> bool:
        is_inside = self.is_inside(cell)
        if is_inside:
            return self.layout[cell[0]][cell[1]] == OPEN_CELL
        return False

    def get_open_cells(self) -> tuple[Cell, ...]:
        cells = self._collect_open_cells()
        return tuple(cells)

    def _collect_open_cells(self) -> list[Cell]:
        cells: list[Cell] = []
        for row in range(len(self.layout)):
            cells.extend(self._row_open_cells(row))
        return cells

    def _row_open_cells(self, row: int) -> list[Cell]:
        columns = range(len(self.layout[row]))
        return [(row, column) for column in columns
                if self.is_open((row, column))]


### Geometry and Adjacency

In [5]:
def move_cell(cell: Cell, direction: Direction) -> Cell:
    row_delta, column_delta = direction.value
    row = cell[0] + row_delta
    column = cell[1] + column_delta
    return row, column


def calculate_manhattan(first: Cell, second: Cell) -> int:
    row_distance = abs(first[0] - second[0])
    column_distance = abs(first[1] - second[1])
    return row_distance + column_distance


def is_available(maze: Maze, cell: Cell,
                 blocked: frozenset[Cell]) -> bool:
    is_blocked = cell in blocked
    if is_blocked:
        return False
    return maze.is_open(cell)


def get_neighbors(maze: Maze, cell: Cell,
                  blocked: frozenset[Cell]) -> tuple[Cell, ...]:
    cells = (move_cell(cell, direction) for direction in Direction)
    return tuple(item for item in cells
                 if is_available(maze, item, blocked))


### Search Context

In [6]:
import heapq
from dataclasses import dataclass, field

FrontierEntry = tuple[int, int, Cell]


@dataclass
class SearchContext:
    frontier: list[FrontierEntry] = field(default_factory=list)
    best_costs: dict[Cell, int] = field(default_factory=dict)
    parents: dict[Cell, Cell] = field(default_factory=dict)
    explored: set[Cell] = field(default_factory=set)


def create_context(start: Cell, task: SearchTask) -> SearchContext:
    context = SearchContext()
    context.best_costs[start] = 0
    priority = calculate_manhattan(start, task.goal)
    heapq.heappush(context.frontier, (priority, 0, start))
    return context


def reconstruct_path(context: SearchContext,
                     goal: Cell) -> tuple[Cell, ...]:
    path = [goal]
    current = goal
    while current in context.parents:
        current = context.parents[current]
        path.append(current)
    return tuple(reversed(path))


def build_result(context: SearchContext,
                 goal: Cell) -> SearchResult:
    path = reconstruct_path(context, goal)
    cost = len(path) - 1
    explored = frozenset(context.explored)
    return SearchResult(goal, path, explored, cost)


def is_current(entry: FrontierEntry, context: SearchContext) -> bool:
    _, cost, cell = entry
    best_cost = context.best_costs.get(cell)
    return best_cost == cost


def has_better_cost(context: SearchContext, edge: Edge) -> bool:
    new_cost = context.best_costs[edge.parent] + 1
    old_cost = context.best_costs.get(edge.neighbor, INFINITE_COST)
    return new_cost < old_cost


def update_neighbor(context: SearchContext, edge: Edge,
                    task: SearchTask) -> None:
    new_cost = context.best_costs[edge.parent] + 1
    context.best_costs[edge.neighbor] = new_cost
    context.parents[edge.neighbor] = edge.parent
    estimate = calculate_manhattan(edge.neighbor, task.goal)
    item = (new_cost + estimate, new_cost, edge.neighbor)
    heapq.heappush(context.frontier, item)


### A* Algorithm

In [7]:
import heapq


def relax_neighbor(context: SearchContext, edge: Edge,
                   task: SearchTask) -> None:
    is_better = has_better_cost(context, edge)
    if is_better:
        update_neighbor(context, edge, task)


def relax_neighbors(context: SearchContext, cell: Cell,
                    task: SearchTask) -> None:
    neighbors = get_neighbors(task.maze, cell, task.blocked)
    for neighbor in neighbors:
        relax_neighbor(context, Edge(cell, neighbor), task)


def get_goal_result(context: SearchContext, cell: Cell,
                    task: SearchTask) -> SearchResult | None:
    is_goal = cell == task.goal
    if is_goal:
        return build_result(context, task.goal)
    return None


def expand_current(context: SearchContext, entry: FrontierEntry,
                   task: SearchTask) -> SearchResult | None:
    context.explored.add(entry[2])
    result = get_goal_result(context, entry[2], task)
    if result:
        return result
    relax_neighbors(context, entry[2], task)
    return None


def process_frontier(context: SearchContext,
                     task: SearchTask) -> SearchResult | None:
    entry = heapq.heappop(context.frontier)
    is_valid = is_current(entry, context)
    if is_valid:
        return expand_current(context, entry, task)
    return None


def run_search(context: SearchContext,
               task: SearchTask) -> SearchResult | None:
    while context.frontier:
        result = process_frontier(context, task)
        if result is not None:
            return result
    return None


def find_path(start: Cell, task: SearchTask) -> SearchResult | None:
    context = create_context(start, task)
    return run_search(context, task)


### Ghost Safety

In [8]:
def get_ghost_zone(maze: Maze, ghost: Cell) -> frozenset[Cell]:
    neighbors = get_neighbors(maze, ghost, frozenset())
    return frozenset((*neighbors, ghost))


def get_danger_cells(maze: Maze,
                     ghosts: tuple[Cell, ...]) -> frozenset[Cell]:
    zones = tuple(get_ghost_zone(maze, ghost) for ghost in ghosts)
    return frozenset().union(*zones)


def calculate_clearance(cell: Cell, ghosts: tuple[Cell, ...]) -> int:
    distances = tuple(calculate_manhattan(cell, ghost)
                      for ghost in ghosts)
    return min(distances, default=0)


### Targeting

In [9]:
def append_result(plans: list[SearchResult],
                  result: SearchResult | None) -> None:
    has_result = result is not None
    if has_result:
        plans.append(result)


def get_food_plans(maze: Maze, state: GameState,
                   blocked: frozenset[tuple[int, int]]
                   ) -> tuple[SearchResult, ...]:
    plans: list[SearchResult] = []
    for food in state.foods:
        task = SearchTask(maze, food, blocked)
        append_result(plans, find_path(state.pacman, task))
    return tuple(plans)


def get_plan_cost(plan: SearchResult) -> int:
    return plan.cost


def choose_food_plan(maze: Maze, state: GameState,
                     blocked: frozenset[tuple[int, int]]
                     ) -> SearchResult | None:
    plans = get_food_plans(maze, state, blocked)
    has_plan = len(plans) > 0
    if has_plan:
        return min(plans, key=get_plan_cost)
    return None


### Controller

In [10]:
from functools import partial


def get_plan_step(plan: SearchResult) -> Cell:
    return plan.path[1]


def choose_escape_cell(maze: Maze, state: GameState,
                       blocked: frozenset[Cell]) -> Cell:
    candidates = get_neighbors(maze, state.pacman, blocked)
    has_candidate = len(candidates) > 0
    if has_candidate:
        key = partial(calculate_clearance, ghosts=state.ghosts)
        return max(candidates, key=key)
    return state.pacman


def choose_pacman_cell(maze: Maze, state: GameState) -> Cell:
    blocked = get_danger_cells(maze, state.ghosts)
    plan = choose_food_plan(maze, state, blocked)
    has_plan = plan is not None
    if has_plan:
        return get_plan_step(plan)
    return choose_escape_cell(maze, state, blocked)


### Ghost Movement

In [11]:
import numpy as np

Rng = np.random.Generator


def choose_ghost_cell(maze: Maze, ghost: Cell, rng: Rng) -> Cell:
    candidates = get_neighbors(maze, ghost, frozenset())
    index = int(rng.integers(len(candidates)))
    return candidates[index]


def move_ghosts(maze: Maze, ghosts: tuple[Cell, ...],
                rng: Rng) -> tuple[Cell, ...]:
    return tuple(choose_ghost_cell(maze, ghost, rng)
                 for ghost in ghosts)


### State Factory

In [12]:
import numpy as np

Rng = np.random.Generator


def sample_cells(maze: Maze, count: int, rng: Rng) -> tuple[Cell, ...]:
    cells = maze.get_open_cells()
    indices = rng.choice(len(cells), size=count, replace=False)
    return tuple(cells[int(index)] for index in indices)


def is_safe_spawn(pacman: Cell, ghosts: tuple[Cell, ...]) -> bool:
    distances = tuple(calculate_manhattan(pacman, ghost)
                      for ghost in ghosts)
    return all(distance >= MIN_SPAWN_DISTANCE for distance in distances)


def split_cells(cells: tuple[Cell, ...],
                config: GameConfig) -> tuple[Cell, tuple[Cell, ...],
                                             frozenset[Cell]]:
    pacman = cells[0]
    ghost_end = 1 + config.ghost_count
    ghosts = cells[1:ghost_end]
    foods = frozenset(cells[ghost_end:])
    return pacman, ghosts, foods


def sample_safe_entities(maze: Maze, config: GameConfig,
                         rng: Rng) -> tuple[Cell, tuple[Cell, ...],
                                            frozenset[Cell]]:
    count = 1 + config.ghost_count + config.food_count
    while True:
        entities = split_cells(sample_cells(maze, count, rng), config)
        if is_safe_spawn(entities[0], entities[1]):
            return entities


def create_random_state(maze: Maze, config: GameConfig,
                        rng: Rng) -> GameState:
    pacman, ghosts, foods = sample_safe_entities(maze, config, rng)
    return GameState(pacman, ghosts, foods, 0, 0, GameStatus.RUNNING)


### Game Engine

In [13]:
from dataclasses import replace
import numpy as np

Rng = np.random.Generator


def move_pacman(state: GameState, pacman: Cell) -> GameState:
    has_food = pacman in state.foods
    foods = state.foods - frozenset((pacman,))
    reward = FOOD_SCORE if has_food else STEP_SCORE
    return replace(state, pacman=pacman, foods=foods,
                   score=state.score + reward)


def apply_ghosts(state: GameState,
                 ghosts: tuple[Cell, ...]) -> GameState:
    has_collision = state.pacman in ghosts
    score = state.score + DEATH_SCORE if has_collision else state.score
    status = GameStatus.LOST if has_collision else state.status
    return replace(state, ghosts=ghosts, score=score, status=status)


def finalize_state(state: GameState, config: GameConfig) -> GameState:
    has_won = len(state.foods) == 0
    if has_won:
        return replace(state, status=GameStatus.WON)
    has_timeout = state.steps >= config.max_steps
    if has_timeout:
        return replace(state, status=GameStatus.LOST)
    return state


class GameEngine:
    def __init__(self, maze: Maze, config: GameConfig, rng: Rng) -> None:
        self.maze = maze
        self.config = config
        self.rng = rng

    def step(self, state: GameState) -> GameState:
        is_running = state.status is GameStatus.RUNNING
        if is_running:
            return self._advance(state)
        return state

    def _advance(self, state: GameState) -> GameState:
        pacman = choose_pacman_cell(self.maze, state)
        moved = move_pacman(state, pacman)
        ghosts = move_ghosts(self.maze, state.ghosts, self.rng)
        with_ghosts = apply_ghosts(moved, ghosts)
        stepped = replace(with_ghosts, steps=state.steps + 1)
        return finalize_state(stepped, self.config)


### Simulation Demo Runner

In [14]:
import numpy as np


def run_engine(engine: GameEngine,
               state: GameState) -> tuple[GameState, ...]:
    states = [state]
    while state.status is GameStatus.RUNNING:
        state = engine.step(state)
        states.append(state)
    return tuple(states)


def simulate_game(maze: Maze, config: GameConfig,
                  seed: int | None = None) -> tuple[GameState, ...]:
    rng = np.random.default_rng(seed)
    state = create_random_state(maze, config, rng)
    engine = GameEngine(maze, config, rng)
    return run_engine(engine, state)


### Evaluation Metrics

In [15]:
from statistics import mean

GameOutcome = tuple[bool, int]


def get_steps(maze: Maze, config: GameConfig,
              seed: int) -> GameOutcome:
    states = simulate_game(maze, config, seed)
    final = states[-1]
    has_won = final.status is GameStatus.WON
    return has_won, final.steps


def summarize(results: tuple[GameOutcome, ...]) -> EvaluationSummary:
    games = len(results)
    wins = sum(1 for has_won, _ in results if has_won)
    steps = tuple(step_count for _, step_count in results)
    rate = 100.0 * wins / games
    return EvaluationSummary(
        games, wins, games - wins, rate, mean(steps), max(steps)
    )


def evaluate_games(maze: Maze, config: GameConfig,
                   games: int) -> EvaluationSummary:
    results = tuple(get_steps(maze, config, seed)
                    for seed in range(games))
    return summarize(results)


### Matplotlib Renderer

In [16]:
from dataclasses import dataclass
from matplotlib.axes import Axes


def configure_axis(ax: Axes, maze: Maze) -> None:
    ax.set_xlim(-0.5, len(maze.layout[0]) - 0.5)
    ax.set_ylim(len(maze.layout) - 0.5, -0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")


def draw_maze(ax: Axes, maze: Maze) -> None:
    for row in range(len(maze.layout)):
        draw_maze_row(ax, maze, row)


def draw_maze_row(ax: Axes, maze: Maze, row: int) -> None:
    for column in range(len(maze.layout[row])):
        draw_maze_cell(ax, maze, (row, column))


def draw_maze_cell(ax: Axes, maze: Maze,
                   cell: tuple[int, int]) -> None:
    is_wall = maze.is_open(cell) is False
    if is_wall:
        ax.scatter(cell[1], cell[0], s=WALL_SIZE,
                   marker="s", color=COLOR_WALL)


def draw_food(ax: Axes, state: GameState) -> None:
    for row, column in state.foods:
        ax.scatter(column, row, s=FOOD_SIZE, color=COLOR_FOOD)


def draw_agents(ax: Axes, state: GameState) -> None:
    row, column = state.pacman
    ax.scatter(column, row, s=PACMAN_SIZE, color=COLOR_PACMAN,
               edgecolors=COLOR_TEXT, linewidths=1.5)
    for ghost_row, ghost_column in state.ghosts:
        ax.scatter(ghost_column, ghost_row, s=GHOST_SIZE,
                   color=COLOR_GHOST, marker="D")


def get_title(state: GameState) -> str:
    status = state.status.value.upper()
    return (
        f"Pac-Man A* | {status} | Step {state.steps} | "
        f"Food left {len(state.foods)} | Score {state.score}"
    )


@dataclass
class FrameRenderer:
    ax: Axes
    maze: Maze
    states: tuple[GameState, ...]

    def __call__(self, frame: int) -> tuple[()]:
        self.ax.clear()
        configure_axis(self.ax, self.maze)
        draw_maze(self.ax, self.maze)
        draw_food(self.ax, self.states[frame])
        draw_agents(self.ax, self.states[frame])
        self.ax.set_title(get_title(self.states[frame]))
        return ()


### Animation View

In [17]:
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.axes import Axes
from matplotlib.figure import Figure

Animation = animation.FuncAnimation


def create_figure() -> tuple[Figure, Axes]:
    figure, axis = plt.subplots(figsize=FIGURE_SIZE)
    return figure, axis


def build_animation(fig: Figure, view: FrameRenderer, count: int) -> Animation:
    return Animation(
        fig, view, range(count),
        interval=ANIMATION_INTERVAL_MS, repeat=False
    )


def create_animation(maze: Maze,
                     states: tuple[GameState, ...]) -> Animation:
    figure, axis = create_figure()
    renderer = FrameRenderer(axis, maze, states)
    movie = build_animation(figure, renderer, len(states))
    plt.close(figure)
    return movie


### Unit Tests Definition

In [18]:
import unittest
import numpy as np


class Phase4Tests(unittest.TestCase):
    def test_manhattan_distance(self) -> None:
        self.assertEqual(calculate_manhattan((0, 0), (2, 2)), 4)

    def test_astar_straight_path(self) -> None:
        maze = Maze(("...", "...", "..."))
        task = SearchTask(maze, (0, 2), frozenset())
        result = find_path((0, 0), task)
        self.assertIsNotNone(result)
        self.assertEqual(result.cost, 2)

    def test_astar_detours_around_wall(self) -> None:
        maze = Maze(("...", ".#.", "..."))
        task = SearchTask(maze, (2, 1), frozenset())
        result = find_path((0, 1), task)
        self.assertIsNotNone(result)
        self.assertEqual(result.cost, 4)

    def test_astar_avoids_ghost_cell(self) -> None:
        maze = Maze(("...", "...", "..."))
        task = SearchTask(maze, (0, 2), frozenset(((0, 1),)))
        result = find_path((0, 0), task)
        self.assertIsNotNone(result)
        self.assertGreater(result.cost, 2)

    def test_astar_reports_no_path(self) -> None:
        maze = Maze((".#.", "###", ".#."))
        task = SearchTask(maze, (0, 2), frozenset())
        result = find_path((0, 0), task)
        self.assertIsNone(result)

    def test_random_state_has_requested_entities(self) -> None:
        maze = Maze(MAZE_LAYOUT)
        config = GameConfig(8, 2, 100)
        state = create_random_state(maze, config, np.random.default_rng(7))
        self.assertEqual(len(state.foods), 8)
        self.assertEqual(len(state.ghosts), 2)

    def test_seeded_game_collects_all_food(self) -> None:
        maze = Maze(MAZE_LAYOUT)
        config = GameConfig(12, 2, 320)
        final = simulate_game(maze, config, 7)[-1]
        self.assertEqual(final.status, GameStatus.WON)
        self.assertEqual(len(final.foods), 0)


def run_tests() -> unittest.result.TestResult:
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(Phase4Tests)
    runner = unittest.TextTestRunner(verbosity=2)
    return runner.run(suite)


## 1. Unit tests

The first tests isolate A* on tiny grids before the full game. They verify:

- Manhattan distance;
- straight shortest path;
- detouring around a wall;
- ghost-cell avoidance;
- no-path handling;
- randomized entity creation;
- a complete seeded automated game.

In [19]:
test_result = run_tests()
assert test_result.wasSuccessful()


test_astar_avoids_ghost_cell (__main__.Phase4Tests.test_astar_avoids_ghost_cell) ... ok
test_astar_detours_around_wall (__main__.Phase4Tests.test_astar_detours_around_wall) ... ok
test_astar_reports_no_path (__main__.Phase4Tests.test_astar_reports_no_path) ... ok
test_astar_straight_path (__main__.Phase4Tests.test_astar_straight_path) ... ok
test_manhattan_distance (__main__.Phase4Tests.test_manhattan_distance) ... ok
test_random_state_has_requested_entities (__main__.Phase4Tests.test_random_state_has_requested_entities) ... ok
test_seeded_game_collects_all_food (__main__.Phase4Tests.test_seeded_game_collects_all_food) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.119s

OK


## 2. Beginner 3×3 A* demonstration

This is the assignment's recommended small test before integration.

In [20]:
tiny_maze = Maze(("...", ".#.", "..."))
tiny_task = SearchTask(tiny_maze, (2, 2), frozenset())
tiny_result = find_path((0, 0), tiny_task)

print("Path:", tiny_result.path)
print("Cost:", tiny_result.cost)
print("Explored cells:", len(tiny_result.explored))


Path: ((0, 0), (0, 1), (0, 2), (1, 2), (2, 2))
Cost: 4
Explored cells: 8


## 3. Full-game configuration

The maze is fixed, while game entities are randomized.

- 12 food items.
- 2 moving ghosts.
- 320-step safety cap.
- Ghost safety includes each ghost's current cell plus cells it could enter on
  the next move.
- Pac-Man replans after every state update.

In [21]:
maze = Maze(MAZE_LAYOUT)
config = GameConfig(
    DEFAULT_FOOD_COUNT,
    DEFAULT_GHOST_COUNT,
    DEFAULT_MAX_STEPS,
)

print("Open cells:", len(maze.get_open_cells()))
print("Food:", config.food_count)
print("Ghosts:", config.ghost_count)


Open cells: 80
Food: 12
Ghosts: 2


## 4. Reproducible evaluation

The actual demo below is randomized. This evaluation intentionally uses fixed
seeds so its claim can be reproduced.

A win means Pac-Man collected **all food without colliding with a ghost** before
the step cap.

In [22]:
summary = evaluate_games(maze, config, EVALUATION_GAMES)

print("Games:", summary.games)
print("Wins:", summary.wins)
print("Losses:", summary.losses)
print(f"Success rate: {summary.success_rate:.1f}%")
print(f"Mean steps: {summary.mean_steps:.1f}")
print("Maximum steps:", summary.max_steps)

assert summary.success_rate >= 95.0


Games: 20
Wins: 20
Losses: 0
Success rate: 100.0%
Mean steps: 73.4
Maximum steps: 119


## 5. Randomized automated game

**Rerun the next two cells to generate a new playthrough.**

`seed=None` means NumPy draws fresh entropy, so Pac-Man, food, ghosts and ghost
movement change every playtime. No keyboard input is read anywhere.

In [23]:
DEMO_SEED: int | None = None
game_states = simulate_game(maze, config, DEMO_SEED)
final_state = game_states[-1]

print("Final status:", final_state.status.value)
print("Steps:", final_state.steps)
print("Score:", final_state.score)
print("Food remaining:", len(final_state.foods))


Final status: won
Steps: 62
Score: 70
Food remaining: 0


## 6. Playing animation

The animation is generated from the automated state sequence. Pac-Man is the
yellow circle, ghosts are red diamonds, food is shown as yellow dots, and dark
cells are walls.

In [24]:
from IPython.display import HTML

pacman_animation = create_animation(maze, game_states)
HTML(pacman_animation.to_jshtml())
